# H&E Data Preparation – Tumor Invasion Front

This notebook orchestrates the three-stage data-preparation pipeline using the
reusable modules in `multiplex_pipeline.hne.preprocessing`:

1. **Patch extraction** – tile the whole-slide OME-TIFF and rasterise QuPath
   polygon annotations into per-tile segmentation masks.
2. **Oversampling** – apply geometric augmentations to patches that contain
   invasion-front tissue and decompose each tile into hematoxylin / eosin
   channels.
3. **Class balancing** – remove stroma-only patches until the stroma pixel
   count no longer exceeds the tumor pixel count.
4. **EDA** – explore the final balanced dataset.


In [ ]:
from pathlib import Path

from multiplex_pipeline.hne.preprocessing.patch_extractor import RobustPatchExtractor
from multiplex_pipeline.hne.preprocessing.balancer import (
    oversample_tumor_patches,
    balance_dataset,
)
from multiplex_pipeline.hne.visualization.eda import (
    plot_class_proportions,
    plot_intensity_distributions,
    plot_correlation_matrix,
    plot_scatter_matrix,
    plot_eosin_by_stroma_quartile,
)
from multiplex_pipeline.hne.config import EDA_DIR

## 1 – Configuration

In [ ]:
import os

# Configure paths via environment variables or edit directly here
BASE_IMG_PATH  = Path(os.environ.get("HNE_IMG_DIR", "/path/to/ome_tiff/images"))
OME_FILENAME   = os.environ.get("HNE_OME_FILENAME", "sample.tif")
BASE_ANN_PATH  = Path(os.environ.get("HNE_ANN_DIR", "/path/to/annotations"))
RAW_PATCHES_DIR    = Path(os.environ.get("HNE_PATCHES_DIR",    "/tmp/patches/raw"))
OVERSAMPLED_DIR    = Path(os.environ.get("HNE_OVERSAMPLED_DIR", "/tmp/patches/oversampled"))
BALANCED_DIR       = Path(os.environ.get("HNE_BALANCED_DIR",    "/tmp/patches/balanced"))
EDA_OUTPUT_DIR     = EDA_DIR

## 2 – Patch extraction

In [ ]:
extractor = RobustPatchExtractor(
    img_path=BASE_IMG_PATH,
    ome_filename=OME_FILENAME,
    ann_path=BASE_ANN_PATH,
    output_dir=RAW_PATCHES_DIR,
)
extractor.extract()

## 3 – Oversampling tumor patches + H&E decomposition

In [ ]:
oversample_tumor_patches(
    src_dir=RAW_PATCHES_DIR,
    dst_dir=OVERSAMPLED_DIR,
)

## 4 – Class balancing

In [ ]:
balance_dataset(
    src_dir=OVERSAMPLED_DIR,
    dst_dir=BALANCED_DIR,
)

## 5 – Exploratory data analysis

In [ ]:
import glob
import os
import cv2
import numpy as np
import pandas as pd
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm
from skimage.color import rgb2hed

from multiplex_pipeline.hne.schema import (
    PATCH_ID, PROP_BACKGROUND, PROP_FRONT, PROP_STROMA,
    MEAN_GRAY, MEAN_HEMATOXYLIN, MEAN_EOSIN,
)

class_labels = {0: "background", 1: "front_of_invasion", 2: "stroma"}

def analyze_patch(mpath):
    pid = os.path.splitext(os.path.basename(mpath))[0].split('_')[1]
    mask = cv2.imread(mpath, cv2.IMREAD_UNCHANGED)
    total = mask.size
    counts = {class_labels[c]: int((mask == c).sum()) for c in class_labels}
    props  = {f"prop_{class_labels[c]}": counts[class_labels[c]] / total for c in class_labels}
    img  = cv2.cvtColor(cv2.imread(mpath.replace("mask_", "patch_")), cv2.COLOR_BGR2RGB)
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    hed  = rgb2hed(img / 255.0)
    h, e = hed[:, :, 0], hed[:, :, 1]
    return {
        PATCH_ID: pid, **counts, **props,
        MEAN_GRAY: gray.mean(),
        MEAN_HEMATOXYLIN: np.clip((h - h.min()) / (h.max() - h.min() + 1e-8), 0, 1).mean(),
        MEAN_EOSIN:       np.clip((e - e.min()) / (e.max() - e.min() + 1e-8), 0, 1).mean(),
    }

final_masks = sorted(glob.glob(os.path.join(BALANCED_DIR, "mask_*.png")))
with ThreadPoolExecutor(max_workers=8) as ex:
    futures = [ex.submit(analyze_patch, m) for m in final_masks]
    df = pd.DataFrame([f.result() for f in tqdm(futures, desc="Analyzing patches")])

for c in df.columns:
    if c != PATCH_ID:
        df[c] = pd.to_numeric(df[c])

print(df.describe().T)

In [ ]:
EDA_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

plot_class_proportions(df,          EDA_OUTPUT_DIR / "class_proportions.svg")
plot_intensity_distributions(df,    EDA_OUTPUT_DIR / "intensity_distributions.svg")
plot_correlation_matrix(df,         EDA_OUTPUT_DIR / "correlation_matrix.svg")
plot_eosin_by_stroma_quartile(df,   EDA_OUTPUT_DIR / "eosin_by_stroma_quartile.svg")
plot_scatter_matrix(df,             EDA_OUTPUT_DIR / "scatter_matrix.svg")

print("EDA plots saved to", EDA_OUTPUT_DIR)